# HGCTM Revision — Documentation-Depth Trimmed Bernoulli Sensitivity

This notebook performs an additional robustness test for the reviewer's concern that well-known and extensively studied deposits may have much longer mineral lists.

It starts from the completed presence/absence analysis and:

1. calculates mapped mineral-species count for every deposit;
2. ranks deposits **within each deposit type** by documentation depth;
3. removes the top 10% of the most extensively documented deposits in each type;
4. refits the Bernoulli HGCTM using:
   - Macrostrat lithology;
   - GLiM lithology;
   - equal lithology and age prior scales, 0.30/0.30;
   - K = 7;
   - seeds 42, 123, and 7;
5. compares every trimmed fit with the matching full-cohort Bernoulli fit.

## Why trim within deposit type?

Documentation depth differs among deposit types. A global top-10% removal could disproportionately remove deposit types that naturally have longer mineral lists. Stratified trimming preserves the approximate deposit-type composition while directly testing whether a small set of exceptionally well-documented deposits drives the result.

## Deterministic trimming rule

For each deposit type:

- sort by mapped mineral-species count, descending;
- break ties using present-family count, descending;
- then use Mindat ID, ascending;
- remove `ceil(0.10 × n_type)` deposits.

The removed and retained IDs, ranks, thresholds, and covariate distributions are exported.

## Main scientific question

Does the stronger lithology-associated effect and the main topic structure persist after the most extensively documented deposits are removed?

## Required Kaggle input

Attach only:

- `HGCTM_Presence_Absence_Sensitivity_K7_Outputs.zip`

This archive already contains:

- the 1,237-deposit count and binary matrices;
- the common covariate table;
- documentation-depth information;
- the full-cohort Bernoulli reference fits.

No new manual data preparation is required.

In [ ]:
# Install only packages that are missing.

import importlib.util
import subprocess
import sys

required = {
    "pyro": "pyro-ppl",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])
else:
    print("All required packages are available.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import time
import warnings
import zipfile

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from torch.nn.functional import softmax

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working")
OUT = WORK / "hgctm_documentation_trim_sensitivity"
RUNS_DIR = OUT / "runs"
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"

for folder in [OUT, RUNS_DIR, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# ---------------- SETTINGS ----------------

RUN_MODE = "full"

K = 7
TRIM_FRACTION = 0.10
SIGMA_LITHO = 0.30
SIGMA_AGE = 0.30
LR = 0.005
PRIMARY_SEED = 42
FULL_SEEDS = [42, 123, 7]
TOP_N_FAMILIES = 10

if RUN_MODE == "quick":
    ACTIVE_SEEDS = [42]
    N_STEPS = 400
    ANNEAL_END = 120
    PRINT_EVERY = 100
elif RUN_MODE == "screening":
    ACTIVE_SEEDS = [42]
    N_STEPS = 2500
    ANNEAL_END = 750
    PRINT_EVERY = 500
elif RUN_MODE == "full":
    ACTIVE_SEEDS = FULL_SEEDS
    N_STEPS = 5000
    ANNEAL_END = 1500
    PRINT_EVERY = 500
else:
    raise ValueError("RUN_MODE must be quick, screening, or full.")

SOURCES = {
    "Macrostrat": "Macrostrat_model_lithology",
    "GLiM": "GLiM_model_lithology",
}

AGE_LEVELS = [
    "Cenozoic",
    "Mesozoic",
    "Paleozoic",
    "Precambrian",
    "Unknown",
]

EXPECTED_FULL_N = 1237
EXPECTED_F = 35

print("Run mode:", RUN_MODE)
print("Trim fraction:", TRIM_FRACTION)
print("Seeds:", ACTIVE_SEEDS)
print("Planned fits:", len(SOURCES) * len(ACTIVE_SEEDS))


## 1. Locate and extract the completed presence/absence output

In [ ]:
def extract_zip_once(zip_path, target):
    target = Path(target)
    marker = target / ".extracted_ok"

    if marker.exists():
        return target

    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)

    marker.write_text(
        datetime.now(timezone.utc).isoformat(),
        encoding="utf-8",
    )
    return target


def locate_presence_output():
    required = {
        "common_cohort_count_matrix.csv",
        "common_cohort_binary_matrix.csv",
        "common_cohort_covariates.csv",
        "presence_absence__bernoulli__macrostrat__equal_high__seed_42.npz",
        "presence_absence__bernoulli__glim__equal_high__seed_42.npz",
    }

    roots = [Path("/kaggle/input"), WORK]

    # Search extracted/direct folders.
    for root in roots:
        if not root.exists():
            continue

        for count_path in root.rglob("common_cohort_count_matrix.csv"):
            candidate_root = count_path.parent.parent
            available = {
                path.name
                for path in candidate_root.rglob("*")
                if path.is_file()
            }
            if required.issubset(available):
                return candidate_root

    # Search archives.
    for root in roots:
        if not root.exists():
            continue

        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name
                        for name in archive.namelist()
                    }
                    if required.issubset(basenames):
                        digest = hashlib.sha256(
                            str(zip_path).encode("utf-8")
                        ).hexdigest()[:12]
                        target = WORK / f"presence_output_{digest}"
                        extract_zip_once(zip_path, target)

                        matches = list(
                            target.rglob(
                                "common_cohort_count_matrix.csv"
                            )
                        )
                        if not matches:
                            continue
                        return matches[0].parent.parent
            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError(
        "Could not locate HGCTM_Presence_Absence_Sensitivity_K7_Outputs.zip "
        "or its extracted contents."
    )


PRESENCE_ROOT = locate_presence_output()
TABLES_INPUT = PRESENCE_ROOT / "tables"
RUNS_INPUT = PRESENCE_ROOT / "runs"

print("Presence/absence output root:", PRESENCE_ROOT)
print("Input tables:", TABLES_INPUT)
print("Input runs:", RUNS_INPUT)


## 2. Load and verify the full common cohort

In [ ]:
def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


count_matrix = pd.read_csv(
    TABLES_INPUT / "common_cohort_count_matrix.csv"
)
binary_matrix = pd.read_csv(
    TABLES_INPUT / "common_cohort_binary_matrix.csv"
)
cohort = pd.read_csv(
    TABLES_INPUT / "common_cohort_covariates.csv"
)

for frame, label in [
    (count_matrix, "count matrix"),
    (binary_matrix, "binary matrix"),
    (cohort, "cohort"),
]:
    if "Mindat_id" not in frame.columns:
        raise KeyError(f"{label} has no Mindat_id column.")
    frame["Mindat_id"] = frame["Mindat_id"].map(canonical_id)
    if frame["Mindat_id"].duplicated().any():
        raise ValueError(f"{label} has duplicated Mindat IDs.")

count_matrix = count_matrix.set_index("Mindat_id")
binary_matrix = binary_matrix.set_index("Mindat_id")
cohort = cohort.set_index("Mindat_id")

common_ids = [
    mindat_id
    for mindat_id in count_matrix.index
    if mindat_id in binary_matrix.index
    and mindat_id in cohort.index
]

count_matrix = count_matrix.loc[common_ids].copy()
binary_matrix = binary_matrix.loc[common_ids].copy()
cohort = cohort.loc[common_ids].copy()

if count_matrix.shape != (EXPECTED_FULL_N, EXPECTED_F):
    raise ValueError(
        f"Expected count matrix ({EXPECTED_FULL_N}, {EXPECTED_F}), "
        f"found {count_matrix.shape}."
    )

if binary_matrix.shape != count_matrix.shape:
    raise ValueError(
        "Binary and count matrices have different shapes."
    )

recomputed_binary = (count_matrix > 0).astype(np.float32)

if not np.array_equal(
    recomputed_binary.to_numpy(),
    binary_matrix.to_numpy(),
):
    raise ValueError(
        "Stored binary matrix does not equal count_matrix > 0."
    )

required_covariates = {
    "Deposit_type",
    "age_category",
    "Macrostrat_model_lithology",
    "GLiM_model_lithology",
}

missing_covariates = required_covariates - set(cohort.columns)

if missing_covariates:
    raise KeyError(
        f"Missing cohort columns: {sorted(missing_covariates)}"
    )

print("Verified full cohort:", count_matrix.shape)
print("Deposit types:")
print(cohort["Deposit_type"].value_counts().to_string())


## 3. Construct the deterministic within-type top-10% trim

The primary documentation-depth measure is the sum of mapped mineral-species counts across the 35 families.

Ties are resolved using:

1. present-family count, descending;
2. Mindat ID, ascending.

In [ ]:
trim_table = cohort[
    [
        "Deposit_type",
        "age_category",
        "Macrostrat_model_lithology",
        "GLiM_model_lithology",
    ]
].copy()

trim_table["mapped_species_count"] = (
    count_matrix.sum(axis=1)
)
trim_table["present_family_count"] = (
    binary_matrix.sum(axis=1)
)
trim_table["Mindat_id"] = trim_table.index

ranking_parts = []
trim_summary_rows = []

for deposit_type, group in trim_table.groupby(
    "Deposit_type",
    sort=True,
):
    group = group.copy()

    group["_id_numeric"] = pd.to_numeric(
        group["Mindat_id"],
        errors="coerce",
    )
    group["_id_text"] = group["Mindat_id"].astype(str)

    group = group.sort_values(
        by=[
            "mapped_species_count",
            "present_family_count",
            "_id_numeric",
            "_id_text",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
        na_position="last",
        kind="mergesort",
    )

    n_type = len(group)
    n_remove = int(
        math.ceil(TRIM_FRACTION * n_type)
    )

    group["documentation_rank_within_type"] = (
        np.arange(1, n_type + 1)
    )
    group["documentation_percentile_within_type"] = (
        group["documentation_rank_within_type"]
        / n_type
    )
    group["remove_top_documented"] = (
        group["documentation_rank_within_type"]
        <= n_remove
    )

    removed = group[
        group["remove_top_documented"]
    ]
    retained = group[
        ~group["remove_top_documented"]
    ]

    trim_summary_rows.append({
        "Deposit_type": deposit_type,
        "n_full": n_type,
        "n_removed": n_remove,
        "n_retained": n_type - n_remove,
        "removed_percent": 100 * n_remove / n_type,
        "minimum_species_count_removed": int(
            removed["mapped_species_count"].min()
        ),
        "maximum_species_count_retained": int(
            retained["mapped_species_count"].max()
        ),
        "removed_species_count_mean": float(
            removed["mapped_species_count"].mean()
        ),
        "retained_species_count_mean": float(
            retained["mapped_species_count"].mean()
        ),
        "removed_family_count_mean": float(
            removed["present_family_count"].mean()
        ),
        "retained_family_count_mean": float(
            retained["present_family_count"].mean()
        ),
    })

    ranking_parts.append(
        group.drop(
            columns=["_id_numeric", "_id_text"]
        )
    )

trim_rankings = pd.concat(
    ranking_parts,
    axis=0,
).sort_index()

trim_summary = pd.DataFrame(
    trim_summary_rows
)

removed_ids = trim_rankings.loc[
    trim_rankings["remove_top_documented"],
    "Mindat_id",
].tolist()

retained_ids = [
    mindat_id
    for mindat_id in common_ids
    if mindat_id not in set(removed_ids)
]

X_counts_trim = count_matrix.loc[
    retained_ids
].copy()

X_binary_trim = binary_matrix.loc[
    retained_ids
].astype(np.float32).copy()

cohort_trim = cohort.loc[
    retained_ids
].copy()

print("Full N:", len(common_ids))
print("Removed N:", len(removed_ids))
print("Retained N:", len(retained_ids))
print()
print(trim_summary.round(3).to_string(index=False))

trim_rankings.to_csv(
    TABLE_DIR / "documentation_trim_rankings_and_status.csv"
)
trim_summary.to_csv(
    TABLE_DIR / "documentation_trim_summary_by_deposit_type.csv",
    index=False,
)

trim_rankings.loc[
    trim_rankings["remove_top_documented"]
].to_csv(
    TABLE_DIR / "removed_high_documentation_deposits.csv"
)

trim_rankings.loc[
    ~trim_rankings["remove_top_documented"]
].to_csv(
    TABLE_DIR / "retained_trimmed_deposits.csv"
)

X_counts_trim.to_csv(
    TABLE_DIR / "trimmed_count_matrix.csv"
)
X_binary_trim.to_csv(
    TABLE_DIR / "trimmed_binary_matrix.csv"
)
cohort_trim.to_csv(
    TABLE_DIR / "trimmed_covariates.csv"
)


## 4. Audit whether trimming changed age or lithology distributions

In [ ]:
def category_distribution_comparison(
    full_series,
    trimmed_series,
    variable,
    source=None,
):
    levels = sorted(
        set(full_series.dropna().astype(str))
        | set(trimmed_series.dropna().astype(str))
    )

    full_counts = (
        full_series.fillna("Missing")
        .astype(str)
        .value_counts()
        .reindex(levels, fill_value=0)
    )
    trimmed_counts = (
        trimmed_series.fillna("Missing")
        .astype(str)
        .value_counts()
        .reindex(levels, fill_value=0)
    )

    full_prop = full_counts / full_counts.sum()
    trimmed_prop = trimmed_counts / trimmed_counts.sum()

    rows = []

    for level in levels:
        rows.append({
            "source": source,
            "variable": variable,
            "category": level,
            "full_count": int(full_counts[level]),
            "trimmed_count": int(trimmed_counts[level]),
            "full_percent": 100 * full_prop[level],
            "trimmed_percent": 100 * trimmed_prop[level],
            "percentage_point_change": (
                100
                * (
                    trimmed_prop[level]
                    - full_prop[level]
                )
            ),
        })

    total_variation = 0.5 * np.abs(
        trimmed_prop - full_prop
    ).sum()

    return rows, float(total_variation)


distribution_rows = []
distribution_distance_rows = []

variables = {
    "Deposit_type": (
        cohort["Deposit_type"],
        cohort_trim["Deposit_type"],
        None,
    ),
    "age_category": (
        cohort["age_category"],
        cohort_trim["age_category"],
        None,
    ),
    "Macrostrat_lithology": (
        cohort["Macrostrat_model_lithology"],
        cohort_trim["Macrostrat_model_lithology"],
        "Macrostrat",
    ),
    "GLiM_lithology": (
        cohort["GLiM_model_lithology"],
        cohort_trim["GLiM_model_lithology"],
        "GLiM",
    ),
}

for variable, (
    full_series,
    trimmed_series,
    source,
) in variables.items():
    rows, distance = (
        category_distribution_comparison(
            full_series,
            trimmed_series,
            variable,
            source,
        )
    )
    distribution_rows.extend(rows)
    distribution_distance_rows.append({
        "source": source,
        "variable": variable,
        "total_variation_distance": distance,
    })

distribution_comparison = pd.DataFrame(
    distribution_rows
)
distribution_distances = pd.DataFrame(
    distribution_distance_rows
)

distribution_comparison.to_csv(
    TABLE_DIR / "full_vs_trimmed_covariate_distributions.csv",
    index=False,
)
distribution_distances.to_csv(
    TABLE_DIR / "full_vs_trimmed_distribution_distances.csv",
    index=False,
)

print("Distribution total-variation distances:")
print(
    distribution_distances.round(4).to_string(
        index=False
    )
)


## 5. Encode the trimmed data and fit a trimmed binary LDA warm start

In [ ]:
all_lithology_levels = sorted(
    set(
        cohort[
            "Macrostrat_model_lithology"
        ].dropna().astype(str)
    )
    | set(
        cohort[
            "GLiM_model_lithology"
        ].dropna().astype(str)
    )
)

lithology_to_index = {
    label: index
    for index, label in enumerate(
        all_lithology_levels
    )
}

age_to_index = {
    label: index
    for index, label in enumerate(AGE_LEVELS)
}

age_values_trim = (
    cohort_trim["age_category"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

unexpected_age = (
    set(age_values_trim.unique())
    - set(AGE_LEVELS)
)

if unexpected_age:
    raise ValueError(
        f"Unexpected age labels: {sorted(unexpected_age)}"
    )

age_idx_trim_np = age_values_trim.map(
    age_to_index
).to_numpy(dtype=np.int64)

age_tensor_trim = torch.tensor(
    age_idx_trim_np,
    dtype=torch.long,
)

source_lithology_idx_trim = {}
source_lithology_tensors_trim = {}

for source_name, column in SOURCES.items():
    values = (
        cohort_trim[column]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    unexpected = (
        set(values.unique())
        - set(all_lithology_levels)
    )

    if unexpected:
        raise ValueError(
            f"Unexpected {source_name} classes: "
            f"{sorted(unexpected)}"
        )

    indices = values.map(
        lithology_to_index
    ).to_numpy(dtype=np.int64)

    source_lithology_idx_trim[
        source_name
    ] = indices

    source_lithology_tensors_trim[
        source_name
    ] = torch.tensor(
        indices,
        dtype=torch.long,
    )

X_binary_trim_np = X_binary_trim.to_numpy(
    dtype=np.float32
)

X_binary_trim_tensor = torch.tensor(
    X_binary_trim_np,
    dtype=torch.float32,
)

N_TRIM, F = X_binary_trim_np.shape
L = len(all_lithology_levels)
A = len(AGE_LEVELS)

print(
    f"Trimmed model dimensions: "
    f"N={N_TRIM}, F={F}, K={K}, L={L}, A={A}"
)
print("Lithology levels:", all_lithology_levels)


In [ ]:
lda_trim = LatentDirichletAllocation(
    n_components=K,
    doc_topic_prior=0.5 / K,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)

lda_trim.fit(
    X_binary_trim_np.astype(np.float64)
)

phi_lda_trim = (
    lda_trim.components_
    / lda_trim.components_.sum(
        axis=1,
        keepdims=True,
    )
)

theta_lda_trim = lda_trim.transform(
    X_binary_trim_np.astype(np.float64)
)

np.save(
    OUT / "phi_lda_trimmed_binary.npy",
    phi_lda_trim,
)
np.save(
    OUT / "theta_lda_trimmed_binary.npy",
    theta_lda_trim,
)

pd.DataFrame(
    phi_lda_trim,
    columns=X_binary_trim.columns,
).to_csv(
    TABLE_DIR / "phi_lda_trimmed_binary.csv",
    index=False,
)

print("Trimmed binary LDA fitted:", phi_lda_trim.shape)


## 6. Bernoulli HGCTM

In [ ]:
class HGCTM_Bernoulli(nn.Module):
    def __init__(
        self,
        K,
        F,
        L,
        A,
        sigma_mu=2.0,
        sigma_litho=0.30,
        sigma_age=0.30,
    ):
        super().__init__()
        self.K = K
        self.F = F
        self.L = L
        self.A = A
        self.sigma_mu = sigma_mu
        self.sigma_litho = sigma_litho
        self.sigma_age = sigma_age

    def model(
        self,
        X,
        lithology_idx,
        age_idx,
    ):
        N_local, F_local = X.shape
        K_local = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(
                torch.zeros(
                    K_local,
                    F_local,
                ),
                self.sigma_mu
                * torch.ones(
                    K_local,
                    F_local,
                ),
            ).to_event(2),
        )

        delta_litho = pyro.sample(
            "delta_litho",
            dist.Normal(
                torch.zeros(
                    K_local,
                    self.L,
                    F_local,
                ),
                self.sigma_litho
                * torch.ones(
                    K_local,
                    self.L,
                    F_local,
                ),
            ).to_event(3),
        )

        delta_age = pyro.sample(
            "delta_age",
            dist.Normal(
                torch.zeros(
                    K_local,
                    self.A,
                    F_local,
                ),
                self.sigma_age
                * torch.ones(
                    K_local,
                    self.A,
                    F_local,
                ),
            ).to_event(3),
        )

        omega_a = pyro.sample(
            "omega_a",
            dist.Beta(2.0, 2.0),
        )

        tau = pyro.sample(
            "tau",
            dist.Beta(
                torch.ones(F_local),
                3.0 * torch.ones(F_local),
            ).to_event(1),
        )

        B = pyro.sample(
            "B",
            dist.Normal(
                torch.zeros(
                    self.L + self.A,
                    K_local - 1,
                ),
                torch.ones(
                    self.L + self.A,
                    K_local - 1,
                ),
            ).to_event(2),
        )

        intercept = pyro.sample(
            "intercept",
            dist.Normal(
                torch.zeros(K_local - 1),
                torch.ones(K_local - 1),
            ).to_event(1),
        )

        with pyro.plate(
            "deposits",
            N_local,
        ):
            z_lithology = (
                torch.nn.functional.one_hot(
                    lithology_idx,
                    self.L,
                ).float()
            )

            z_age = (
                torch.nn.functional.one_hot(
                    age_idx,
                    self.A,
                ).float()
            )

            z = torch.cat(
                [z_lithology, z_age],
                dim=1,
            )

            eta = z @ B + intercept
            eta_full = torch.cat(
                [
                    eta,
                    torch.zeros(
                        N_local,
                        1,
                    ),
                ],
                dim=1,
            )
            theta = softmax(
                eta_full,
                dim=1,
            )

            lithology_deviation = (
                delta_litho[
                    :,
                    lithology_idx,
                    :,
                ].permute(1, 0, 2)
            )

            age_deviation = (
                delta_age[
                    :,
                    age_idx,
                    :,
                ].permute(1, 0, 2)
            )

            psi = (
                mu.unsqueeze(0)
                + tau.view(1, 1, -1)
                * lithology_deviation
                + omega_a
                * tau.view(1, 1, -1)
                * age_deviation
            )

            topic_family_probability = (
                torch.sigmoid(psi)
            )

            p = torch.einsum(
                "nk,nkf->nf",
                theta,
                topic_family_probability,
            )

            p = p.clamp(
                min=1e-6,
                max=1.0 - 1e-6,
            )

            pyro.sample(
                "obs",
                dist.Bernoulli(
                    probs=p
                ).to_event(1),
                obs=X,
            )

    def guide(
        self,
        X,
        lithology_idx,
        age_idx,
    ):
        _, F_local = X.shape
        K_local = self.K

        mu_loc = pyro.param(
            "mu_loc",
            torch.randn(
                K_local,
                F_local,
            ) * 0.1,
        )
        mu_scale = pyro.param(
            "mu_scale",
            0.5
            * torch.ones(
                K_local,
                F_local,
            ),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        pyro.sample(
            "mu",
            dist.Normal(
                mu_loc,
                mu_scale,
            ).to_event(2),
        )

        delta_litho_loc = pyro.param(
            "delta_litho_loc",
            torch.zeros(
                K_local,
                self.L,
                F_local,
            ),
        )
        delta_litho_scale = pyro.param(
            "delta_litho_scale",
            0.2
            * torch.ones(
                K_local,
                self.L,
                F_local,
            ),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        pyro.sample(
            "delta_litho",
            dist.Normal(
                delta_litho_loc,
                delta_litho_scale,
            ).to_event(3),
        )

        delta_age_loc = pyro.param(
            "delta_age_loc",
            torch.zeros(
                K_local,
                self.A,
                F_local,
            ),
        )
        delta_age_scale = pyro.param(
            "delta_age_scale",
            0.1
            * torch.ones(
                K_local,
                self.A,
                F_local,
            ),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        pyro.sample(
            "delta_age",
            dist.Normal(
                delta_age_loc,
                delta_age_scale,
            ).to_event(3),
        )

        omega_a_a = pyro.param(
            "omega_a_a",
            torch.tensor(2.0),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        omega_a_b = pyro.param(
            "omega_a_b",
            torch.tensor(2.0),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        pyro.sample(
            "omega_a",
            dist.Beta(
                omega_a_a,
                omega_a_b,
            ),
        )

        tau_a = pyro.param(
            "tau_a",
            torch.ones(F_local),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        tau_b = pyro.param(
            "tau_b",
            3.0 * torch.ones(F_local),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        pyro.sample(
            "tau",
            dist.Beta(
                tau_a,
                tau_b,
            ).to_event(1),
        )

        B_loc = pyro.param(
            "B_loc",
            torch.zeros(
                self.L + self.A,
                K_local - 1,
            ),
        )
        B_scale = pyro.param(
            "B_scale",
            0.5
            * torch.ones(
                self.L + self.A,
                K_local - 1,
            ),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        pyro.sample(
            "B",
            dist.Normal(
                B_loc,
                B_scale,
            ).to_event(2),
        )

        intercept_loc = pyro.param(
            "intercept_loc",
            torch.zeros(K_local - 1),
        )
        intercept_scale = pyro.param(
            "intercept_scale",
            0.5
            * torch.ones(K_local - 1),
            constraint=(
                torch.distributions
                .constraints.positive
            ),
        )
        pyro.sample(
            "intercept",
            dist.Normal(
                intercept_loc,
                intercept_scale,
            ).to_event(1),
        )


## 7. Warm start, fitting, and extraction utilities

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.set_rng_seed(seed)


def logit(values):
    values = np.clip(
        values,
        1e-5,
        1.0 - 1e-5,
    )
    return np.log(
        values / (1.0 - values)
    )


def make_trimmed_bernoulli_warm_start():
    marginal_prevalence = np.clip(
        X_binary_trim_np.mean(axis=0),
        0.01,
        0.99,
    )

    log_phi = np.log(
        np.clip(
            phi_lda_trim,
            1e-6,
            None,
        )
    )

    topic_signal = (
        log_phi
        - log_phi.mean(
            axis=0,
            keepdims=True,
        )
    )

    mu_init = (
        logit(
            marginal_prevalence
        )[None, :]
        + 0.75 * topic_signal
    )

    return torch.tensor(
        mu_init,
        dtype=torch.float32,
    )


def fit_model(
    model,
    X,
    lithology_tensor,
    age_tensor,
    seed,
):
    set_all_seeds(seed)
    pyro.clear_param_store()

    optimizer = ClippedAdam({
        "lr": LR,
        "betas": (0.9, 0.999),
    })

    svi = SVI(
        model.model,
        model.guide,
        optimizer,
        loss=Trace_ELBO(),
    )

    # Create parameters.
    svi.step(
        X,
        lithology_tensor,
        age_tensor,
    )

    pyro.param(
        "mu_loc"
    ).data.copy_(
        make_trimmed_bernoulli_warm_start()
    )

    pyro.param(
        "tau_a"
    ).data.copy_(
        50.0 * torch.ones(
            X.shape[1]
        )
    )
    pyro.param(
        "tau_b"
    ).data.copy_(
        torch.ones(
            X.shape[1]
        )
    )

    losses = []

    for step in range(N_STEPS):
        loss = svi.step(
            X,
            lithology_tensor,
            age_tensor,
        )
        losses.append(float(loss))

        if step < ANNEAL_END:
            progress = (
                step / ANNEAL_END
            )
            minimum_tau_a = (
                50.0
                * (1.0 - progress)
                + progress
            )

            with torch.no_grad():
                pyro.param(
                    "tau_a"
                ).data.clamp_(
                    min=minimum_tau_a
                )

        if (
            step + 1
        ) % PRINT_EVERY == 0:
            tau_a = pyro.param(
                "tau_a"
            ).detach()
            tau_b = pyro.param(
                "tau_b"
            ).detach()
            tau_mean = (
                tau_a
                / (tau_a + tau_b)
            ).mean().item()

            print(
                f"    step {step+1:>5d} "
                f"loss={loss:>12.1f} "
                f"tau_mean={tau_mean:.3f}"
            )

    return losses


def extract_parameters():
    mu = pyro.param(
        "mu_loc"
    ).detach().cpu().numpy()

    delta_litho = pyro.param(
        "delta_litho_loc"
    ).detach().cpu().numpy()

    delta_age = pyro.param(
        "delta_age_loc"
    ).detach().cpu().numpy()

    B = pyro.param(
        "B_loc"
    ).detach().cpu().numpy()

    intercept = pyro.param(
        "intercept_loc"
    ).detach().cpu().numpy()

    tau_a = pyro.param(
        "tau_a"
    ).detach().cpu().numpy()
    tau_b = pyro.param(
        "tau_b"
    ).detach().cpu().numpy()

    tau = (
        tau_a
        / (tau_a + tau_b)
    )

    omega_a_a = float(
        pyro.param(
            "omega_a_a"
        ).item()
    )
    omega_a_b = float(
        pyro.param(
            "omega_a_b"
        ).item()
    )

    omega_a = (
        omega_a_a
        / (
            omega_a_a
            + omega_a_b
        )
    )

    topic_probability = (
        1.0
        / (
            1.0
            + np.exp(-mu)
        )
    )

    topic_profile_normalized = (
        topic_probability
        / topic_probability.sum(
            axis=1,
            keepdims=True,
        )
    )

    return {
        "mu": mu,
        "delta_litho": delta_litho,
        "delta_age": delta_age,
        "B": B,
        "intercept": intercept,
        "tau": tau,
        "omega_a": omega_a,
        "topic_probability": (
            topic_probability
        ),
        "topic_profile_normalized": (
            topic_profile_normalized
        ),
    }


## 8. Deterministic outputs and comparison metrics

In [ ]:
def deterministic_outputs(
    params,
    lithology_tensor,
    age_tensor,
):
    mu = torch.tensor(
        params["mu"],
        dtype=torch.float32,
    )
    delta_litho = torch.tensor(
        params["delta_litho"],
        dtype=torch.float32,
    )
    delta_age = torch.tensor(
        params["delta_age"],
        dtype=torch.float32,
    )
    B = torch.tensor(
        params["B"],
        dtype=torch.float32,
    )
    intercept = torch.tensor(
        params["intercept"],
        dtype=torch.float32,
    )
    tau = torch.tensor(
        params["tau"],
        dtype=torch.float32,
    )
    omega_a = torch.tensor(
        params["omega_a"],
        dtype=torch.float32,
    )

    with torch.no_grad():
        z_lithology = (
            torch.nn.functional.one_hot(
                lithology_tensor,
                L,
            ).float()
        )

        z_age = (
            torch.nn.functional.one_hot(
                age_tensor,
                A,
            ).float()
        )

        z = torch.cat(
            [z_lithology, z_age],
            dim=1,
        )

        eta = z @ B + intercept
        eta_full = torch.cat(
            [
                eta,
                torch.zeros(
                    len(lithology_tensor),
                    1,
                ),
            ],
            dim=1,
        )
        theta = softmax(
            eta_full,
            dim=1,
        )

        lithology_deviation = (
            delta_litho[
                :,
                lithology_tensor,
                :,
            ].permute(1, 0, 2)
        )

        age_deviation = (
            delta_age[
                :,
                age_tensor,
                :,
            ].permute(1, 0, 2)
        )

        psi = (
            mu.unsqueeze(0)
            + tau.view(1, 1, -1)
            * lithology_deviation
            + omega_a
            * tau.view(1, 1, -1)
            * age_deviation
        )

        topic_family_probability = (
            torch.sigmoid(psi)
        )

        p = torch.einsum(
            "nk,nkf->nf",
            theta,
            topic_family_probability,
        )

        p = p.clamp(
            min=1e-6,
            max=1.0 - 1e-6,
        )

    return {
        "theta": theta.cpu().numpy(),
        "topic_family_context": (
            topic_family_probability
            .cpu()
            .numpy()
        ),
        "p": p.cpu().numpy(),
    }


def bernoulli_log_prob(
    X,
    p,
):
    X_t = torch.tensor(
        X,
        dtype=torch.float64,
    )
    p_t = torch.tensor(
        p,
        dtype=torch.float64,
    ).clamp(
        min=1e-12,
        max=1.0 - 1e-12,
    )

    per_family = (
        X_t * torch.log(p_t)
        + (1.0 - X_t)
        * torch.log(1.0 - p_t)
    )

    return (
        per_family.sum(
            dim=1
        ).cpu().numpy(),
        per_family.cpu().numpy(),
    )


def brier_score(
    X,
    p,
):
    return float(
        np.mean(
            (X - p) ** 2
        )
    )


def cosine_similarity(
    a,
    b,
):
    denominator = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(a, b)
        / denominator
    )


def align_topics_to_reference(
    run_profile,
    reference_profile,
):
    similarity = np.zeros(
        (K, K)
    )

    for run_topic in range(K):
        for reference_topic in range(K):
            similarity[
                run_topic,
                reference_topic,
            ] = cosine_similarity(
                run_profile[run_topic],
                reference_profile[
                    reference_topic
                ],
            )

    run_indices, reference_indices = (
        linear_sum_assignment(
            -similarity
        )
    )

    reference_to_run = np.full(
        K,
        -1,
        dtype=int,
    )

    for (
        run_topic,
        reference_topic,
    ) in zip(
        run_indices,
        reference_indices,
    ):
        reference_to_run[
            reference_topic
        ] = run_topic

    topic_cosines = np.array([
        similarity[
            reference_to_run[
                reference_topic
            ],
            reference_topic,
        ]
        for reference_topic
        in range(K)
    ])

    return {
        "reference_to_run": (
            reference_to_run
        ),
        "aligned_profile": (
            run_profile[
                reference_to_run
            ]
        ),
        "topic_cosines": (
            topic_cosines
        ),
        "similarity_matrix": (
            similarity
        ),
    }


def top_family_jaccard(
    profile_1,
    profile_2,
    top_n=10,
):
    values = []

    for topic in range(K):
        top_1 = set(
            np.argsort(
                profile_1[topic]
            )[::-1][:top_n]
        )
        top_2 = set(
            np.argsort(
                profile_2[topic]
            )[::-1][:top_n]
        )

        values.append(
            len(top_1 & top_2)
            / len(top_1 | top_2)
        )

    return np.array(values)


def weighted_average(
    values,
    counts,
    include_mask,
):
    values = np.asarray(
        values,
        dtype=float,
    )
    counts = np.asarray(
        counts,
        dtype=float,
    )
    include_mask = np.asarray(
        include_mask,
        dtype=bool,
    )

    usable = (
        include_mask
        & (counts > 0)
        & np.isfinite(values)
    )

    if not usable.any():
        return np.nan

    return float(
        np.average(
            values[usable],
            weights=counts[usable],
        )
    )


def compute_effect_metrics(
    params,
    lithology_indices,
    age_indices,
):
    tau = params["tau"]

    lithology_class_effects = np.mean(
        np.abs(
            params[
                "delta_litho"
            ]
        )
        * tau[
            None,
            None,
            :,
        ],
        axis=(0, 2),
    )

    age_class_effects = np.mean(
        np.abs(
            params[
                "delta_age"
            ]
        )
        * tau[
            None,
            None,
            :,
        ]
        * params["omega_a"],
        axis=(0, 2),
    )

    lithology_counts = np.bincount(
        lithology_indices,
        minlength=L,
    )

    age_counts = np.bincount(
        age_indices,
        minlength=A,
    )

    lithology_nonmissing = np.array([
        label != "unknown"
        for label
        in all_lithology_levels
    ])

    lithology_strict = np.array([
        label not in {
            "other",
            "unknown",
        }
        for label
        in all_lithology_levels
    ])

    age_nonmissing = np.array([
        label != "Unknown"
        for label in AGE_LEVELS
    ])

    lithology_nonmissing_effect = (
        weighted_average(
            lithology_class_effects,
            lithology_counts,
            lithology_nonmissing,
        )
    )

    age_nonmissing_effect = (
        weighted_average(
            age_class_effects,
            age_counts,
            age_nonmissing,
        )
    )

    lithology_strict_effect = (
        weighted_average(
            lithology_class_effects,
            lithology_counts,
            lithology_strict,
        )
    )

    age_strict_effect = (
        weighted_average(
            age_class_effects,
            age_counts,
            age_nonmissing,
        )
    )

    def safe_ratio(
        numerator,
        denominator,
    ):
        if (
            not np.isfinite(
                numerator
            )
            or not np.isfinite(
                denominator
            )
            or denominator <= 0
        ):
            return np.nan

        return float(
            numerator
            / denominator
        )

    metrics = {
        "lithology_effect_weighted_nonmissing": (
            lithology_nonmissing_effect
        ),
        "age_effect_weighted_nonmissing": (
            age_nonmissing_effect
        ),
        "ratio_weighted_nonmissing_primary": (
            safe_ratio(
                lithology_nonmissing_effect,
                age_nonmissing_effect,
            )
        ),
        "lithology_effect_weighted_strict": (
            lithology_strict_effect
        ),
        "age_effect_weighted_strict": (
            age_strict_effect
        ),
        "ratio_weighted_strict": (
            safe_ratio(
                lithology_strict_effect,
                age_strict_effect,
            )
        ),
        "tau_mean": float(
            np.mean(tau)
        ),
        "tau_median": float(
            np.median(tau)
        ),
    }

    detail_rows = []

    for (
        index,
        label,
    ) in enumerate(
        all_lithology_levels
    ):
        detail_rows.append({
            "covariate": "lithology",
            "class_index": index,
            "class_label": label,
            "count": int(
                lithology_counts[
                    index
                ]
            ),
            "class_effect": float(
                lithology_class_effects[
                    index
                ]
            ),
            "strict_resolved": (
                label not in {
                    "other",
                    "unknown",
                }
            ),
        })

    for (
        index,
        label,
    ) in enumerate(
        AGE_LEVELS
    ):
        detail_rows.append({
            "covariate": "age",
            "class_index": index,
            "class_label": label,
            "count": int(
                age_counts[
                    index
                ]
            ),
            "class_effect": float(
                age_class_effects[
                    index
                ]
            ),
            "strict_resolved": (
                label != "Unknown"
            ),
        })

    return (
        metrics,
        pd.DataFrame(
            detail_rows
        ),
    )


## 9. Load matching full-cohort Bernoulli reference runs

In [ ]:
def full_run_path(
    source_name,
    seed,
):
    stem = (
        "presence_absence"
        "__bernoulli"
        f"__{source_name.lower()}"
        "__equal_high"
        f"__seed_{seed}"
    )
    return RUNS_INPUT / f"{stem}.npz"


def full_summary_path(
    source_name,
    seed,
):
    stem = (
        "presence_absence"
        "__bernoulli"
        f"__{source_name.lower()}"
        "__equal_high"
        f"__seed_{seed}"
    )
    return (
        RUNS_INPUT
        / f"{stem}_run_summary.json"
    )


def load_full_reference(
    source_name,
    seed,
):
    npz_path = full_run_path(
        source_name,
        seed,
    )

    if not npz_path.exists():
        raise FileNotFoundError(
            f"Missing full reference: {npz_path}"
        )

    archive = np.load(
        npz_path,
        allow_pickle=True,
    )

    full_ids = [
        canonical_id(value)
        for value in archive[
            "Mindat_id"
        ].tolist()
    ]

    id_to_position = {
        mindat_id: position
        for position, mindat_id
        in enumerate(full_ids)
    }

    missing = [
        mindat_id
        for mindat_id
        in retained_ids
        if mindat_id
        not in id_to_position
    ]

    if missing:
        raise ValueError(
            "Full reference does not contain all retained IDs. "
            f"First missing: {missing[:10]}"
        )

    retained_order = np.array([
        id_to_position[
            mindat_id
        ]
        for mindat_id
        in retained_ids
    ])

    summary = None
    summary_path = full_summary_path(
        source_name,
        seed,
    )

    if summary_path.exists():
        summary = json.loads(
            summary_path.read_text(
                encoding="utf-8"
            )
        )

    return {
        "topic_profile": archive[
            "topic_profile_normalized"
        ],
        "theta_retained": archive[
            "theta"
        ][retained_order],
        "fitted_probabilities_retained": archive[
            "fitted_binary_probabilities"
        ][retained_order],
        "summary": summary,
        "path": str(npz_path),
    }


for source_name in SOURCES:
    for seed in ACTIVE_SEEDS:
        reference = load_full_reference(
            source_name,
            seed,
        )
        print(
            source_name,
            seed,
            reference[
                "topic_profile"
            ].shape,
            reference[
                "theta_retained"
            ].shape,
        )


## 10. Resumable run helpers

In [ ]:
def run_stem(
    source_name,
    seed,
):
    return (
        "documentation_trim"
        "__bernoulli"
        f"__{source_name.lower()}"
        "__equal_high"
        f"__seed_{seed}"
    )


def existing_summary(
    source_name,
    seed,
):
    path = (
        RUNS_DIR
        / f"{run_stem(source_name, seed)}"
        "_run_summary.json"
    )

    if not path.exists():
        return None

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def import_resume_runs():
    imported = 0

    for root in [
        Path("/kaggle/input"),
        WORK,
    ]:
        if not root.exists():
            continue

        for zip_path in root.rglob(
            "*.zip"
        ):
            try:
                with zipfile.ZipFile(
                    zip_path
                ) as archive:
                    names = archive.namelist()
                    if any(
                        "documentation_trim"
                        in Path(
                            name
                        ).name
                        and name.endswith(
                            "_run_summary.json"
                        )
                        for name in names
                    ):
                        digest = (
                            hashlib.sha256(
                                str(
                                    zip_path
                                ).encode(
                                    "utf-8"
                                )
                            )
                            .hexdigest()[:12]
                        )
                        target = (
                            WORK
                            / f"trim_resume_{digest}"
                        )
                        extract_zip_once(
                            zip_path,
                            target,
                        )
            except zipfile.BadZipFile:
                continue

    for root in list(
        Path("/kaggle/input").rglob("*")
    ) + list(
        WORK.glob(
            "trim_resume_*"
        )
    ):
        if not root.is_dir():
            continue

        for summary_path in root.rglob(
            "documentation_trim"
            "*_run_summary.json"
        ):
            destination = (
                RUNS_DIR
                / summary_path.name
            )

            if destination.exists():
                continue

            stem = (
                summary_path.name
                .replace(
                    "_run_summary.json",
                    "",
                )
            )

            for source_file in (
                summary_path.parent
                .glob(f"{stem}*")
            ):
                if source_file.is_file():
                    shutil.copy2(
                        source_file,
                        RUNS_DIR
                        / source_file.name,
                    )

            imported += 1

    return imported


print(
    "Imported resume runs:",
    import_resume_runs(),
)


## 11. Fit six trimmed Bernoulli models

In [ ]:
summaries = []

for source_name in SOURCES:
    lithology_indices = (
        source_lithology_idx_trim[
            source_name
        ]
    )

    lithology_tensor = (
        source_lithology_tensors_trim[
            source_name
        ]
    )

    for seed in ACTIVE_SEEDS:
        cached = existing_summary(
            source_name,
            seed,
        )

        if cached is not None:
            print(
                "SKIP completed run:",
                run_stem(
                    source_name,
                    seed,
                ),
            )
            summaries.append(cached)
            continue

        print("\n" + "=" * 80)
        print(
            f"SOURCE={source_name} | "
            f"PRIOR=equal_high | "
            f"SEED={seed}"
        )
        print("=" * 80)

        model = HGCTM_Bernoulli(
            K=K,
            F=F,
            L=L,
            A=A,
            sigma_mu=2.0,
            sigma_litho=SIGMA_LITHO,
            sigma_age=SIGMA_AGE,
        )

        start = time.time()

        losses = fit_model(
            model=model,
            X=X_binary_trim_tensor,
            lithology_tensor=(
                lithology_tensor
            ),
            age_tensor=(
                age_tensor_trim
            ),
            seed=seed,
        )

        elapsed = (
            time.time()
            - start
        )

        params = extract_parameters()

        outputs = deterministic_outputs(
            params,
            lithology_tensor,
            age_tensor_trim,
        )

        (
            log_prob_per_document,
            per_family_log_prob,
        ) = bernoulli_log_prob(
            X_binary_trim_np,
            outputs["p"],
        )

        effects, effect_details = (
            compute_effect_metrics(
                params,
                lithology_indices,
                age_idx_trim_np,
            )
        )

        full_reference = (
            load_full_reference(
                source_name,
                seed,
            )
        )

        alignment = (
            align_topics_to_reference(
                params[
                    "topic_profile_normalized"
                ],
                full_reference[
                    "topic_profile"
                ],
            )
        )

        aligned_profile = alignment[
            "aligned_profile"
        ]

        aligned_theta = outputs[
            "theta"
        ][:, alignment[
            "reference_to_run"
        ]]

        top_jaccard = top_family_jaccard(
            aligned_profile,
            full_reference[
                "topic_profile"
            ],
            top_n=TOP_N_FAMILIES,
        )

        dominant_agreement = float(
            np.mean(
                np.argmax(
                    aligned_theta,
                    axis=1,
                )
                == np.argmax(
                    full_reference[
                        "theta_retained"
                    ],
                    axis=1,
                )
            )
        )

        theta_mae = float(
            np.mean(
                np.abs(
                    aligned_theta
                    - full_reference[
                        "theta_retained"
                    ]
                )
            )
        )

        full_summary = (
            full_reference[
                "summary"
            ]
        )

        full_ratio = (
            full_summary.get(
                "ratio_weighted_nonmissing_primary"
            )
            if full_summary
            is not None
            else np.nan
        )

        full_strict_ratio = (
            full_summary.get(
                "ratio_weighted_strict"
            )
            if full_summary
            is not None
            else np.nan
        )

        marginal_probability = np.clip(
            X_binary_trim_np.mean(
                axis=0,
                keepdims=True,
            ),
            1e-6,
            1.0 - 1e-6,
        )

        marginal_probability = (
            np.repeat(
                marginal_probability,
                N_TRIM,
                axis=0,
            )
        )

        (
            marginal_log_prob,
            _,
        ) = bernoulli_log_prob(
            X_binary_trim_np,
            marginal_probability,
        )

        summary = {
            "analysis": (
                "documentation_depth_trimmed_bernoulli"
            ),
            "source": source_name,
            "seed": int(seed),
            "run_mode": RUN_MODE,
            "trim_fraction": (
                TRIM_FRACTION
            ),
            "n_full": int(
                len(common_ids)
            ),
            "n_removed": int(
                len(removed_ids)
            ),
            "n_retained": int(
                N_TRIM
            ),
            "sigma_litho": (
                SIGMA_LITHO
            ),
            "sigma_age": (
                SIGMA_AGE
            ),
            "n_steps": int(
                len(losses)
            ),
            "elapsed_seconds": float(
                elapsed
            ),
            "omega_a": float(
                params["omega_a"]
            ),
            "tau_mean": float(
                effects["tau_mean"]
            ),
            "elbo_mean_last_100_per_deposit": float(
                -np.mean(
                    losses[-100:]
                )
                / N_TRIM
            ),
            "log_prob_mean_per_deposit": float(
                np.mean(
                    log_prob_per_document
                )
            ),
            "log_prob_per_binary_cell": float(
                np.sum(
                    log_prob_per_document
                )
                / (
                    N_TRIM
                    * F
                )
            ),
            "brier_score": brier_score(
                X_binary_trim_np,
                outputs["p"],
            ),
            "marginal_baseline_log_prob_per_binary_cell": float(
                np.sum(
                    marginal_log_prob
                )
                / (
                    N_TRIM
                    * F
                )
            ),
            "marginal_baseline_brier_score": brier_score(
                X_binary_trim_np,
                marginal_probability,
            ),
            **{
                key: float(value)
                for key, value
                in effects.items()
            },
            "full_ratio_weighted_nonmissing_primary": (
                float(full_ratio)
                if full_ratio
                is not None
                and np.isfinite(
                    full_ratio
                )
                else None
            ),
            "full_ratio_weighted_strict": (
                float(
                    full_strict_ratio
                )
                if full_strict_ratio
                is not None
                and np.isfinite(
                    full_strict_ratio
                )
                else None
            ),
            "trim_minus_full_primary_ratio": (
                float(
                    effects[
                        "ratio_weighted_nonmissing_primary"
                    ]
                    - full_ratio
                )
                if full_ratio
                is not None
                and np.isfinite(
                    full_ratio
                )
                else None
            ),
            "mean_topic_cosine_to_full": float(
                alignment[
                    "topic_cosines"
                ].mean()
            ),
            "minimum_topic_cosine_to_full": float(
                alignment[
                    "topic_cosines"
                ].min()
            ),
            "mean_top10_jaccard_to_full": float(
                top_jaccard.mean()
            ),
            "minimum_top10_jaccard_to_full": float(
                top_jaccard.min()
            ),
            "dominant_topic_agreement_on_retained_deposits": (
                dominant_agreement
            ),
            "mean_absolute_theta_difference_from_full": (
                theta_mae
            ),
            "topic7_cosine_to_full": float(
                alignment[
                    "topic_cosines"
                ][6]
            ),
            "topic7_top10_jaccard_to_full": float(
                top_jaccard[6]
            ),
        }

        stem = run_stem(
            source_name,
            seed,
        )

        np.savez_compressed(
            RUNS_DIR / f"{stem}.npz",
            analysis=np.array(
                summary[
                    "analysis"
                ]
            ),
            source=np.array(
                source_name
            ),
            seed=np.array(seed),
            retained_Mindat_id=np.array(
                retained_ids
            ),
            removed_Mindat_id=np.array(
                removed_ids
            ),
            family_names=np.array(
                X_binary_trim.columns
            ),
            lithology_levels=np.array(
                all_lithology_levels
            ),
            age_levels=np.array(
                AGE_LEVELS
            ),
            losses=np.asarray(
                losses
            ),
            mu=params["mu"],
            delta_litho=params[
                "delta_litho"
            ],
            delta_age=params[
                "delta_age"
            ],
            B=params["B"],
            intercept=params[
                "intercept"
            ],
            tau=params["tau"],
            omega_a=np.array(
                params["omega_a"]
            ),
            topic_probability=params[
                "topic_probability"
            ],
            topic_profile_normalized=params[
                "topic_profile_normalized"
            ],
            topic_profile_aligned_to_full=(
                aligned_profile
            ),
            topic_cosines_to_full=(
                alignment[
                    "topic_cosines"
                ]
            ),
            top10_jaccard_to_full=(
                top_jaccard
            ),
            theta_raw=outputs[
                "theta"
            ],
            theta_aligned_to_full=(
                aligned_theta
            ),
            fitted_binary_probabilities=(
                outputs["p"]
            ),
            log_prob_per_document=(
                log_prob_per_document
            ),
            per_family_log_prob=(
                per_family_log_prob
            ),
            reference_to_run=(
                alignment[
                    "reference_to_run"
                ]
            ),
        )

        pd.DataFrame({
            "step": np.arange(
                1,
                len(losses) + 1,
            ),
            "loss": losses,
            "elbo": -np.asarray(
                losses
            ),
        }).to_csv(
            RUNS_DIR
            / f"{stem}_loss_trace.csv",
            index=False,
        )

        effect_details.to_csv(
            RUNS_DIR
            / f"{stem}_class_effects.csv",
            index=False,
        )

        with open(
            RUNS_DIR
            / f"{stem}_run_summary.json",
            "w",
            encoding="utf-8",
        ) as handle:
            json.dump(
                summary,
                handle,
                indent=2,
                ensure_ascii=False,
            )

        summaries.append(summary)

        print(
            f"Completed {stem}: "
            f"trimmed ratio="
            f"{summary['ratio_weighted_nonmissing_primary']:.3f}; "
            f"full ratio="
            f"{summary['full_ratio_weighted_nonmissing_primary']:.3f}; "
            f"topic cosine="
            f"{summary['mean_topic_cosine_to_full']:.3f}"
        )

summary_df = pd.DataFrame(
    summaries
)

summary_df.to_csv(
    TABLE_DIR
    / "all_documentation_trim_run_summary.csv",
    index=False,
)

print("\nCompleted runs:", len(summary_df))
print(
    summary_df[
        [
            "source",
            "seed",
            "n_retained",
            "ratio_weighted_nonmissing_primary",
            "full_ratio_weighted_nonmissing_primary",
            "ratio_weighted_strict",
            "mean_topic_cosine_to_full",
            "minimum_topic_cosine_to_full",
            "dominant_topic_agreement_on_retained_deposits",
            "topic7_cosine_to_full",
        ]
    ].round(4).to_string(
        index=False
    )
)


## 12. Aggregate results across seeds

In [ ]:
metrics = [
    "ratio_weighted_nonmissing_primary",
    "ratio_weighted_strict",
    "full_ratio_weighted_nonmissing_primary",
    "full_ratio_weighted_strict",
    "trim_minus_full_primary_ratio",
    "omega_a",
    "tau_mean",
    "log_prob_per_binary_cell",
    "brier_score",
    "marginal_baseline_log_prob_per_binary_cell",
    "marginal_baseline_brier_score",
    "mean_topic_cosine_to_full",
    "minimum_topic_cosine_to_full",
    "mean_top10_jaccard_to_full",
    "minimum_top10_jaccard_to_full",
    "dominant_topic_agreement_on_retained_deposits",
    "mean_absolute_theta_difference_from_full",
    "topic7_cosine_to_full",
    "topic7_top10_jaccard_to_full",
]

aggregate_rows = []

for source_name, group in (
    summary_df.groupby(
        "source",
        sort=True,
    )
):
    row = {
        "source": source_name,
        "n_seeds": int(
            len(group)
        ),
        "n_full": int(
            group["n_full"].iloc[0]
        ),
        "n_removed": int(
            group[
                "n_removed"
            ].iloc[0]
        ),
        "n_retained": int(
            group[
                "n_retained"
            ].iloc[0]
        ),
    }

    for metric in metrics:
        values = pd.to_numeric(
            group[metric],
            errors="coerce",
        )

        row[
            f"{metric}_mean"
        ] = float(
            values.mean()
        )
        row[
            f"{metric}_median"
        ] = float(
            values.median()
        )
        row[
            f"{metric}_min"
        ] = float(
            values.min()
        )
        row[
            f"{metric}_max"
        ] = float(
            values.max()
        )
        row[
            f"{metric}_std"
        ] = float(
            values.std(
                ddof=0
            )
        )

    row[
        "trimmed_primary_ratio_all_seeds_gt_1"
    ] = bool(
        (
            group[
                "ratio_weighted_nonmissing_primary"
            ]
            > 1
        ).all()
    )

    row[
        "trimmed_strict_ratio_all_seeds_gt_1"
    ] = bool(
        (
            group[
                "ratio_weighted_strict"
            ]
            > 1
        ).all()
    )

    aggregate_rows.append(row)

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR
    / "documentation_trim_summary_across_seeds.csv",
    index=False,
)

display_columns = [
    "source",
    "n_removed",
    "n_retained",
    "ratio_weighted_nonmissing_primary_median",
    "ratio_weighted_nonmissing_primary_min",
    "ratio_weighted_nonmissing_primary_max",
    "full_ratio_weighted_nonmissing_primary_median",
    "ratio_weighted_strict_median",
    "mean_topic_cosine_to_full_median",
    "minimum_topic_cosine_to_full_min",
    "dominant_topic_agreement_on_retained_deposits_median",
    "topic7_cosine_to_full_median",
    "trimmed_primary_ratio_all_seeds_gt_1",
    "trimmed_strict_ratio_all_seeds_gt_1",
]

print(
    aggregate_df[
        display_columns
    ].round(4).to_string(
        index=False
    )
)


## 13. Topic stability across trimmed seeds

In [ ]:
def load_trimmed_run(
    source_name,
    seed,
):
    return np.load(
        RUNS_DIR
        / f"{run_stem(source_name, seed)}.npz",
        allow_pickle=True,
    )


seed_stability_rows = []

for source_name in SOURCES:
    reference_seed = (
        PRIMARY_SEED
        if PRIMARY_SEED
        in ACTIVE_SEEDS
        else ACTIVE_SEEDS[0]
    )

    reference_profile = (
        load_trimmed_run(
            source_name,
            reference_seed,
        )[
            "topic_profile_aligned_to_full"
        ]
    )

    for seed in ACTIVE_SEEDS:
        profile = load_trimmed_run(
            source_name,
            seed,
        )[
            "topic_profile_aligned_to_full"
        ]

        for topic in range(K):
            seed_stability_rows.append({
                "source": source_name,
                "reference_seed": (
                    reference_seed
                ),
                "seed": seed,
                "topic": topic + 1,
                "cosine_to_reference_seed": (
                    cosine_similarity(
                        profile[topic],
                        reference_profile[
                            topic
                        ],
                    )
                ),
            })

seed_stability = pd.DataFrame(
    seed_stability_rows
)

seed_stability.to_csv(
    TABLE_DIR
    / "trimmed_within_source_seed_stability.csv",
    index=False,
)

print(
    seed_stability.groupby(
        "source"
    )[
        "cosine_to_reference_seed"
    ].agg(
        ["mean", "min", "std"]
    ).round(4).to_string()
)


## 14. Topic-level and top-family comparison

In [ ]:
topic_rows = []
top_family_rows = []

for source_name in SOURCES:
    for seed in ACTIVE_SEEDS:
        run = load_trimmed_run(
            source_name,
            seed,
        )

        aligned_profile = run[
            "topic_profile_aligned_to_full"
        ]
        topic_cosines = run[
            "topic_cosines_to_full"
        ]
        top_jaccard = run[
            "top10_jaccard_to_full"
        ]

        full_reference = (
            load_full_reference(
                source_name,
                seed,
            )
        )

        full_profile = (
            full_reference[
                "topic_profile"
            ]
        )

        for topic in range(K):
            topic_rows.append({
                "source": source_name,
                "seed": seed,
                "topic": topic + 1,
                "topic_cosine_to_full": float(
                    topic_cosines[
                        topic
                    ]
                ),
                "top10_jaccard_to_full": float(
                    top_jaccard[
                        topic
                    ]
                ),
                "is_topic_7": bool(
                    topic == 6
                ),
            })

            top_trimmed = np.argsort(
                aligned_profile[
                    topic
                ]
            )[::-1][
                :TOP_N_FAMILIES
            ]

            top_full = np.argsort(
                full_profile[
                    topic
                ]
            )[::-1][
                :TOP_N_FAMILIES
            ]

            for rank in range(
                TOP_N_FAMILIES
            ):
                top_family_rows.append({
                    "source": source_name,
                    "seed": seed,
                    "topic": topic + 1,
                    "rank": rank + 1,
                    "trimmed_family": (
                        X_binary_trim.columns[
                            top_trimmed[
                                rank
                            ]
                        ]
                    ),
                    "full_family": (
                        X_binary_trim.columns[
                            top_full[
                                rank
                            ]
                        ]
                    ),
                    "same_at_rank": bool(
                        top_trimmed[
                            rank
                        ]
                        == top_full[
                            rank
                        ]
                    ),
                })

topic_comparison = pd.DataFrame(
    topic_rows
)
top_family_comparison = pd.DataFrame(
    top_family_rows
)

topic_comparison.to_csv(
    TABLE_DIR
    / "trimmed_topic_level_comparison_to_full.csv",
    index=False,
)

top_family_comparison.to_csv(
    TABLE_DIR
    / "trimmed_vs_full_top_family_rankings.csv",
    index=False,
)

print(
    topic_comparison.groupby(
        [
            "source",
            "topic",
        ]
    )[
        [
            "topic_cosine_to_full",
            "top10_jaccard_to_full",
        ]
    ].agg(
        ["mean", "min"]
    ).round(4).to_string()
)


## 15. Reviewer-oriented decision table

In [ ]:
decision_rows = []

for _, row in (
    aggregate_df.iterrows()
):
    ratio_min = row[
        "ratio_weighted_nonmissing_primary_min"
    ]
    strict_min = row[
        "ratio_weighted_strict_min"
    ]
    topic_median = row[
        "mean_topic_cosine_to_full_median"
    ]
    topic_min = row[
        "minimum_topic_cosine_to_full_min"
    ]
    dominant_agreement = row[
        "dominant_topic_agreement_on_retained_deposits_median"
    ]

    if (
        ratio_min > 1
        and strict_min > 1
        and topic_median >= 0.90
        and topic_min >= 0.75
    ):
        conclusion = (
            "strong_support_after_documentation_trim"
        )
    elif (
        ratio_min > 1
        and strict_min > 1
    ):
        conclusion = (
            "effect_direction_robust_but_some_topics_sensitive"
        )
    else:
        conclusion = (
            "documentation_trim_changes_effect_direction"
        )

    decision_rows.append({
        "source": row["source"],
        "n_removed": int(
            row["n_removed"]
        ),
        "n_retained": int(
            row["n_retained"]
        ),
        "primary_ratio_min": ratio_min,
        "strict_ratio_min": strict_min,
        "mean_topic_cosine_median": (
            topic_median
        ),
        "minimum_topic_cosine_min": (
            topic_min
        ),
        "dominant_topic_agreement_median": (
            dominant_agreement
        ),
        "topic7_cosine_median": row[
            "topic7_cosine_to_full_median"
        ],
        "conclusion": conclusion,
    })

decision_table = pd.DataFrame(
    decision_rows
)

decision_table.to_csv(
    TABLE_DIR
    / "reviewer_documentation_trim_decision_table.csv",
    index=False,
)

print(
    decision_table.round(4).to_string(
        index=False
    )
)


## 16. Figures

In [ ]:
# Figure 1: documentation-depth distributions.
fig, ax = plt.subplots(
    figsize=(9, 5)
)

ax.hist(
    trim_rankings.loc[
        ~trim_rankings[
            "remove_top_documented"
        ],
        "mapped_species_count",
    ],
    bins=30,
    alpha=0.65,
    label="Retained",
)

ax.hist(
    trim_rankings.loc[
        trim_rankings[
            "remove_top_documented"
        ],
        "mapped_species_count",
    ],
    bins=30,
    alpha=0.65,
    label="Removed top 10% within type",
)

ax.set_xlabel(
    "Mapped mineral-species count"
)
ax.set_ylabel("Deposits")
ax.set_title(
    "Documentation-depth trimming"
)
ax.legend()
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "documentation_depth_removed_vs_retained.png",
    dpi=300,
)
plt.close()


# Figure 2: full versus trimmed effect ratio.
fig, ax = plt.subplots(
    figsize=(8, 5)
)

positions = np.arange(
    len(summary_df)
)

ax.scatter(
    positions,
    summary_df[
        "full_ratio_weighted_nonmissing_primary"
    ],
    label="Full cohort",
)

ax.scatter(
    positions,
    summary_df[
        "ratio_weighted_nonmissing_primary"
    ],
    label="Trimmed cohort",
)

ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
)

ax.set_xticks(positions)
ax.set_xticklabels(
    [
        f"{source}\nseed {seed}"
        for source, seed
        in zip(
            summary_df["source"],
            summary_df["seed"],
        )
    ],
    rotation=35,
    ha="right",
)

ax.set_ylabel(
    "Frequency-weighted lithology / age ratio"
)
ax.set_title(
    "Full versus documentation-trimmed Bernoulli HGCTM"
)
ax.legend()
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "full_vs_trimmed_effect_ratio.png",
    dpi=300,
)
plt.close()


# Figure 3: topic cosine to full reference.
fig, ax = plt.subplots(
    figsize=(10, 5)
)

for source_name in SOURCES:
    subset = topic_comparison[
        topic_comparison[
            "source"
        ]
        == source_name
    ]

    means = (
        subset.groupby(
            "topic"
        )[
            "topic_cosine_to_full"
        ].mean()
    )

    ax.plot(
        means.index,
        means.values,
        marker="o",
        label=source_name,
    )

ax.set_xticks(
    range(1, K + 1)
)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Topic")
ax.set_ylabel(
    "Mean cosine to full-cohort topic"
)
ax.set_title(
    "Topic stability after documentation-depth trimming"
)
ax.legend()
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "topic_stability_after_documentation_trim.png",
    dpi=300,
)
plt.close()


# Figure 4: covariate-distribution shifts.
fig, ax = plt.subplots(
    figsize=(8, 4.5)
)

ax.bar(
    distribution_distances[
        "variable"
    ],
    distribution_distances[
        "total_variation_distance"
    ],
)

ax.set_ylabel(
    "Total-variation distance"
)
ax.set_title(
    "Covariate-distribution change after trimming"
)
ax.tick_params(
    axis="x",
    rotation=30,
)
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "covariate_distribution_shift_after_trim.png",
    dpi=300,
)
plt.close()

print("Saved figures:")
for path in sorted(
    FIG_DIR.iterdir()
):
    print(" -", path.name)


## 17. Provenance and output archive

In [ ]:
manifest = {
    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "analysis": (
        "documentation_depth_trimmed_bernoulli_sensitivity"
    ),
    "notebook": (
        "HGCTM_Documentation_Depth_Trimmed_Bernoulli_K7.ipynb"
    ),
    "input_root": str(
        PRESENCE_ROOT
    ),
    "trim_rule": {
        "fraction": TRIM_FRACTION,
        "stratification": (
            "Deposit_type"
        ),
        "n_removed_rule": (
            "ceil(fraction * n_type)"
        ),
        "sort_order": [
            "mapped_species_count descending",
            "present_family_count descending",
            "Mindat_id ascending",
        ],
    },
    "cohort": {
        "n_full": int(
            len(common_ids)
        ),
        "n_removed": int(
            len(removed_ids)
        ),
        "n_retained": int(
            N_TRIM
        ),
        "n_families": int(F),
    },
    "model": {
        "likelihood": "Bernoulli",
        "K": K,
        "sigma_lithology": (
            SIGMA_LITHO
        ),
        "sigma_age": (
            SIGMA_AGE
        ),
        "tau_prior": "Beta(1,3)",
        "omega_age_prior": (
            "Beta(2,2)"
        ),
        "steps": N_STEPS,
        "anneal_end": (
            ANNEAL_END
        ),
        "learning_rate": LR,
        "seeds": ACTIVE_SEEDS,
        "warm_start": (
            "trimmed binary LDA plus trimmed marginal family prevalence"
        ),
    },
    "comparison": {
        "reference": (
            "matching full-cohort Bernoulli HGCTM, equal-high prior"
        ),
        "metrics": [
            "frequency-weighted lithology-to-age ratio",
            "strict-resolved ratio",
            "topic cosine",
            "top-10 family Jaccard",
            "dominant-topic agreement on retained deposits",
            "mean absolute theta difference",
            "Topic 7 sensitivity",
        ],
    },
}

with open(
    OUT / "run_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )

environment = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "pyro": pyro.__version__,
}

with open(
    OUT / "environment_versions.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        environment,
        handle,
        indent=2,
    )

readme = '''
HGCTM documentation-depth trimmed Bernoulli sensitivity

Purpose
-------
This analysis tests whether the presence/absence results are driven by
a small set of exceptionally well-documented deposits.

Trimming
--------
Within each deposit type, deposits are ranked by:
1. mapped mineral-species count, descending;
2. present mineral-family count, descending;
3. Mindat ID, ascending.

ceil(10% × n_type) deposits are removed from every deposit type.

Model
-----
- Bernoulli HGCTM
- K = 7
- Macrostrat and GLiM
- equal lithology and age prior scales: 0.30/0.30
- seeds 42, 123, and 7
- 5,000 steps in full mode

Comparison
----------
Every trimmed run is aligned with the corresponding full-cohort
Bernoulli run from the previous presence/absence output archive.
'''.strip()

(
    OUT / "README.txt"
).write_text(
    readme,
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(
        WORK
        / "HGCTM_Documentation_Depth_Trimmed_Bernoulli_K7_Outputs"
    ),
    "zip",
    root_dir=OUT,
)

print("Output archive:")
print(archive_path)

print("\nKey tables:")
for path in sorted(
    TABLE_DIR.iterdir()
):
    print(" -", path.name)
